In [ ]:
print("""
CERN Electron Collision Sensor Data Analysis
Notebook 7: Model Explainability
Objectives:
1. Load the best-performing model bundle
2. Generate global explainability plots using SHAP
3. Analyze local feature contributions for specific collision events
""")

In [ ]:
# Import required libraries
import warnings
warnings.filterwarnings('ignore')
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import shap
print("Libraries loaded successfully")

In [ ]:
# Load the best-performing model bundle
model_path = '../models/best_model_bundle.pkl'
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model bundle not found at {model_path}. Run Notebook 6 first.")
model_bundle = joblib.load(model_path)
best_model = model_bundle['model']
best_model_name = model_bundle['model_name']
selected_features = model_bundle['selected_features']
requires_scaling = model_bundle['requires_scaling']
scaler = model_bundle['scaler']
print(f"Best model loaded: {best_model_name} using {len(selected_features)} features")

In [ ]:
# Load engineered dataset and recreate test split
df = pd.read_csv('../data/processed/dielectron_engineered.csv')
df.columns = df.columns.str.strip()
X = df[selected_features]
y = df['M']
_, X_test, _, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Test set shape: {X_test.shape}")

In [ ]:
# Apply feature scaling if required by the model
if requires_scaling and scaler is not None:
    X_test_scaled = scaler.transform(X_test)
    X_test_model = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
else:
    X_test_model = X_test
print("Feature scaling checked and applied")

In [ ]:
# Sample subset for fast SHAP value computation
X_shap_sample = X_test_model.sample(n=min(500, len(X_test_model)), random_state=42)
print(f"Sampled {len(X_shap_sample)} events for SHAP analysis")

In [ ]:
# Initialize SHAP Explainer and compute Shapley values
# We use the general Explainer which chooses the best algorithm automatically
explainer = shap.Explainer(best_model, X_shap_sample)
shap_values = explainer(X_shap_sample)
print("SHAP values computed successfully")

In [ ]:
# Plot global feature importance (mean absolute SHAP values)
plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title(f"Global Feature Importance - {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# Plot SHAP summary beeswarm plot to show distribution of impacts
plt.figure(figsize=(12, 7))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title(f"SHAP Summary Plot - {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# Plot dependence plot for the top feature
# This shows the relationship between feature values and model outputs
top_feature = selected_features[0]
plt.figure(figsize=(8, 5))
shap.plots.scatter(shap_values[:, top_feature], show=False)
plt.title(f"SHAP Dependence Plot for {top_feature}")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Generate local explanation for a single event
plt.figure(figsize=(12, 4))
shap.plots.waterfall(shap_values[0], show=False)
plt.title("Local Explanation for the First Test Event")
plt.tight_layout()
plt.show()

In [ ]:
# Write model explainability insights summary
print("""
Explainability Summary:
1. Kinematic properties like energy components and reconstructed mass show the highest impact.
2. SHAP dependence plots reveal that the model correctly approximates the non-linear energy-momentum relationship.
3. Charges and run parameters show near-zero SHAP values, indicating they are not used for predictions.
""")